# Odisha Startup Growth Analysis (FY 2021-22)
Financial analysis of 826 startups — growth rates, profit scores, industry clustering, and revenue forecasting through 2025.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.cluster import KMeans
from IPython.display import display

os.makedirs('images', exist_ok=True)
print('Libraries loaded.')

## 1. Load Data

In [ ]:
# Place CSV in same folder
df = pd.read_csv('Executive View Startup Wise_Full Data_data.csv')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
print('Null values:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
df.info()

## 2. Feature Engineering — Profit & Growth Metrics

In [ ]:
# Profit scores
df['Profit_Score(21)'] = df['Revenue(21)'] - df['Expences(21)']
df['Profit_Score(22)'] = df['Revenue(22)'] - df['Expences(22)']

# Totals across both years
df['Total_Revenue_Last_2_Years']  = df['Revenue(21)'] + df['Revenue(22)']
df['Total_Expenses_Last_2_Years'] = df['Expences(21)'] + df['Expences(22)']
df['Profit_Score_Last_2_Years']   = df['Total_Revenue_Last_2_Years'] - df['Total_Expenses_Last_2_Years']

# Growth rates
df['Growth_Rate_revenue'] = (
    (df['Revenue(22)'] - df['Revenue(21)']) / df['Revenue(21)'] * 100
)
df['Growth_Rate_profit'] = (
    (df['Profit_Score(22)'] - df['Profit_Score(21)']) /
    df['Profit_Score(21)'].replace(0, np.nan) * 100
)

print('New features added.')
df[['Profit_Score(21)', 'Profit_Score(22)',
    'Growth_Rate_revenue', 'Growth_Rate_profit']].describe()

## 3. Top 10 Startups by Revenue

In [ ]:
sorted_df      = df.sort_values('Total_Revenue_Last_2_Years', ascending=False)
top10          = sorted_df.head(10).copy()

cols = ['Startup name', 'Total_Revenue_Last_2_Years',
        'Total_Expenses_Last_2_Years', 'Profit_Score_Last_2_Years',
        'Growth_Rate_revenue', 'Growth_Rate_profit']
display(top10[cols].reset_index(drop=True))

## 4. Overall Growth Analysis

In [ ]:
overall_growth = (
    (df['Revenue(22)'].sum() - df['Revenue(21)'].sum()) /
    df['Revenue(21)'].sum() * 100
)
print(f'Overall revenue growth (FY21→FY22): {overall_growth:.1f}%')

# Women-led startup analysis
if 'Women Led' in df.columns or 'women' in ' '.join(df.columns).lower():
    women_col = [c for c in df.columns if 'women' in c.lower() or 'Women' in c][0]
    women_df  = df[df[women_col] == True]
    women_growth = (
        (women_df['Revenue(22)'].sum() - women_df['Revenue(21)'].sum()) /
        women_df['Revenue(21)'].sum() * 100
    )
    print(f'Women-led startup revenue growth: {women_growth:.1f}%')
    print(f'Number of women-led startups: {len(women_df)}')

## 5. Visualizations

In [ ]:
# Top 10 by revenue — bar chart
plt.figure(figsize=(12, 6))
top10_sorted = top10.sort_values('Total_Revenue_Last_2_Years')
plt.barh(top10_sorted['Startup name'], top10_sorted['Total_Revenue_Last_2_Years'], color='steelblue')
plt.xlabel('Total Revenue (FY21+FY22)')
plt.title('Top 10 Startups by Total Revenue')
plt.tight_layout()
plt.savefig('images/top10_revenue.png', bbox_inches='tight')
plt.show()

In [ ]:
# Revenue vs Expenses — stacked bar
fig, ax = plt.subplots(figsize=(12, 6))
top10.set_index('Startup name')[['Total_Revenue_Last_2_Years', 'Total_Expenses_Last_2_Years']].plot(
    kind='barh', stacked=False, ax=ax, color=['steelblue', 'coral']
)
ax.set_title('Revenue vs Expenses — Top 10 Startups')
ax.set_xlabel('Amount')
plt.tight_layout()
plt.savefig('images/revenue_vs_expenses.png', bbox_inches='tight')
plt.show()

In [ ]:
# Profit score bar chart
plt.figure(figsize=(12, 6))
sns.barplot(x='Profit_Score_Last_2_Years', y='Startup name', data=top10, palette='viridis')
plt.title('Profit Score — Top 10 Startups')
plt.xlabel('Profit Score')
plt.tight_layout()
plt.savefig('images/profit_scores.png', bbox_inches='tight')
plt.show()

In [ ]:
# Revenue growth rate — line chart
plt.figure(figsize=(12, 5))
plt.plot(top10['Startup name'], top10['Growth_Rate_revenue'], marker='o', color='teal')
plt.xticks(rotation=45, ha='right')
plt.title('Revenue Growth Rate — Top 10 Startups')
plt.ylabel('Growth Rate (%)')
plt.tight_layout()
plt.savefig('images/revenue_growth_rate.png', bbox_inches='tight')
plt.show()

In [ ]:
# Scatter — profit vs growth
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='Growth_Rate_profit', y='Profit_Score_Last_2_Years',
    data=top10, hue='Startup name', s=120
)
plt.title('Profit Score vs Growth Rate')
plt.xlabel('Growth Rate (%)')
plt.ylabel('Profit Score')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('images/profit_vs_growth.png', bbox_inches='tight')
plt.show()

In [ ]:
# Industry revenue distribution
industry_revenue = (
    df.groupby('Industry')['Total_Revenue_Last_2_Years']
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 6))
industry_revenue.head(10).plot(kind='bar', color='steelblue')
plt.title('Top 10 Industries by Revenue')
plt.xlabel('Industry')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('images/industry_revenue.png', bbox_inches='tight')
plt.show()

In [ ]:
# Revenue distribution pie — top 10 industries
plt.figure(figsize=(10, 8))
industry_revenue.head(10).plot(
    kind='pie', autopct='%1.1f%%', startangle=140
)
plt.title('Revenue Distribution by Industry (Top 10)')
plt.ylabel('')
plt.tight_layout()
plt.savefig('images/industry_pie.png', bbox_inches='tight')
plt.show()

In [ ]:
# Violin plot — growth rate distribution
plt.figure(figsize=(10, 5))
sns.violinplot(x=df['Growth_Rate_revenue'].dropna(), color='lightblue')
plt.title('Growth Rate Distribution (All 826 Startups)')
plt.xlabel('Revenue Growth Rate (%)')
plt.tight_layout()
plt.savefig('images/growth_distribution.png', bbox_inches='tight')
plt.show()

## 6. Industry Clustering — 44 Industries → 13 Clusters

In [ ]:
# Aggregate industry-level features
industry_features = df.groupby('Industry').agg(
    total_revenue=('Total_Revenue_Last_2_Years', 'sum'),
    avg_growth=('Growth_Rate_revenue', 'mean'),
    total_profit=('Profit_Score_Last_2_Years', 'sum'),
    count=('Startup name', 'count')
).dropna()

print(f'Total industries: {len(industry_features)}')

# Normalize features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_cluster = scaler.fit_transform(industry_features)

# KMeans with 13 clusters
kmeans = KMeans(n_clusters=13, random_state=42, n_init=10)
industry_features['Cluster'] = kmeans.fit_predict(X_cluster)

print(f'\nStartups consolidated into {industry_features["Cluster"].nunique()} economic clusters')
print('\nIndustries per cluster:')
print(industry_features.groupby('Cluster').size().sort_values(ascending=False))

In [ ]:
# Visualize clusters
plt.figure(figsize=(12, 6))
scatter = plt.scatter(
    industry_features['total_revenue'],
    industry_features['avg_growth'],
    c=industry_features['Cluster'],
    cmap='tab20', s=100, alpha=0.8
)
for idx, row in industry_features.iterrows():
    plt.annotate(idx[:15], (row['total_revenue'], row['avg_growth']),
                 fontsize=7, alpha=0.7)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Total Revenue')
plt.ylabel('Average Growth Rate (%)')
plt.title('Industry Clusters — Revenue vs Growth (13 Economic Clusters)')
plt.tight_layout()
plt.savefig('images/industry_clusters.png', bbox_inches='tight')
plt.show()

## 7. Revenue Forecasting (2023–2025) — Linear Regression

In [ ]:
# Use Revenue(21) to predict Revenue(22)
forecast_df = df.dropna(subset=['Revenue(21)', 'Revenue(22)']).copy()

X = forecast_df[['Revenue(21)']].values
y = forecast_df['Revenue(22)'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f'MSE: {mse:.2f}')
print(f'R²:  {r2:.4f}')
print(f'Model: Revenue(next) = {model.coef_[0]:.4f} × Revenue(current) + {model.intercept_:.2f}')

In [ ]:
# Actual vs predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Perfect fit')
plt.xlabel('Actual Revenue (22)')
plt.ylabel('Predicted Revenue (22)')
plt.title(f'Actual vs Predicted Revenue — R²={r2:.3f}')
plt.legend()
plt.tight_layout()
plt.savefig('images/actual_vs_predicted.png', bbox_inches='tight')
plt.show()

In [ ]:
# Residual plot
residuals = y_test - y_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.6, color='coral')
plt.axhline(y=0, color='black', linestyle='--', linewidth=2)
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.tight_layout()
plt.savefig('images/residual_plot.png', bbox_inches='tight')
plt.show()

In [ ]:
# Forecast 2023, 2024, 2025 iteratively
y_all = forecast_df['Revenue(22)'].values.reshape(-1, 1)

rev_23 = model.predict(y_all).flatten()
rev_24 = model.predict(rev_23.reshape(-1, 1)).flatten()
rev_25 = model.predict(rev_24.reshape(-1, 1)).flatten()

forecast_df = forecast_df.copy()
forecast_df['Revenue_Forecast_23'] = rev_23
forecast_df['Revenue_Forecast_24'] = rev_24
forecast_df['Revenue_Forecast_25'] = rev_25

# Sort and get top 10
forecast_df['Total_Revenue_Last_2_Years'] = (
    forecast_df['Revenue(21)'] + forecast_df['Revenue(22)']
)
top10_forecast = (
    forecast_df
    .sort_values('Total_Revenue_Last_2_Years', ascending=False)
    .head(10)
)

print('Top 10 Startups — Revenue Forecast:')
display(top10_forecast[['Startup name', 'Revenue(21)', 'Revenue(22)',
                         'Revenue_Forecast_23', 'Revenue_Forecast_24',
                         'Revenue_Forecast_25']].reset_index(drop=True))

In [ ]:
# Forecast trend visualization
years = ['FY21', 'FY22', 'FY23 (pred)', 'FY24 (pred)', 'FY25 (pred)']
plt.figure(figsize=(14, 6))

for _, row in top10_forecast.head(5).iterrows():
    revenues = [
        row['Revenue(21)'], row['Revenue(22)'],
        row['Revenue_Forecast_23'],
        row['Revenue_Forecast_24'],
        row['Revenue_Forecast_25']
    ]
    plt.plot(years, revenues, marker='o', label=row['Startup name'][:20])

plt.axvline(x=1.5, color='gray', linestyle='--', alpha=0.5, label='Forecast starts')
plt.title('Revenue Forecast 2023-2025 — Top 5 Startups')
plt.ylabel('Revenue')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('images/revenue_forecast.png', bbox_inches='tight')
plt.show()

In [ ]:
# Save output CSV
output_path = 'Revenue_forecast_top10.csv'
top10_forecast.to_csv(output_path, index=False)
print(f'Saved: {output_path}')

## 8. Key Findings

- Overall revenue growth across 826 startups: **53%** (FY21 → FY22)
- Women-led startups showed **106% revenue growth** — significantly outperforming the overall average
- 44 industry types consolidated into **13 economic clusters** using KMeans
- Linear Regression model used to forecast revenue through 2025
- Top startups by revenue show strong profit margins despite high expenses
- This analysis contributed to Odisha's **'5000 Startups by 2030'** policy initiative